# Loan Core — Kaggle CPU preflight

This notebook checks environment feasibility only. It starts PostgreSQL 17 and one compiled identity service, checks health, and stops both. It does not produce a load benchmark. Backend services, PostgreSQL, and the later Locust generator share notebook resources.

Create a Kaggle notebook with **Accelerator: None**, **Internet: On**, and **Persistence: No persistence**. If Internet is unavailable, complete the account verification Kaggle requests. Note the CPU/session allowance shown by Kaggle in `QUOTA_NOTE`; do not enter credentials. No Docker, Doppler, or managed database is used. Run all cells and download the ZIP from the output link. See [Kaggle notebook documentation](https://www.kaggle.com/docs/notebooks).


In [ ]:
from pathlib import Path
import os
import subprocess

SOURCE_REF = "refactor/shared-infrastructure"  # Set to the supplied commit SHA for a reproducible run.
QUOTA_NOTE = ""  # Copy the CPU/session allowance shown in your Kaggle UI, or explain that it is not displayed.

assert Path("/kaggle/working").is_dir(), "Use a Kaggle CPU notebook"
assert QUOTA_NOTE.strip(), "Record the allowance observed in the Kaggle UI before continuing"
ROOT = Path("/kaggle/working/loan-core")
TOOLCHAIN = Path("/kaggle/working/loan-core-toolchain")
TOOLCHAIN.mkdir(exist_ok=True)

def run(args, **kwargs):
    subprocess.run(args, check=True, **kwargs)


## Install the pinned tools

Installation needs root or passwordless sudo and Internet access. Node is installed in the notebook's working directory after SHA-256 verification. PostgreSQL comes from the [official PostgreSQL Ubuntu repository](https://www.postgresql.org/download/linux/ubuntu/). The package repository supplies PostgreSQL 17 maintenance updates; the observed exact version is recorded in the report.


In [ ]:
import hashlib
import platform
import shutil
import tarfile
from urllib.request import urlopen

admin = [] if os.geteuid() == 0 else ["sudo", "-n"]
run(admin + ["true"])
run(admin + ["apt-get", "update", "-qq"])
run(admin + ["apt-get", "install", "-y", "--no-install-recommends", "ca-certificates", "curl", "gnupg", "git", "xz-utils", "build-essential", "postgresql-common"])
run(admin + ["/usr/share/postgresql-common/pgdg/apt.postgresql.org.sh", "-y"])
run(admin + ["apt-get", "update", "-qq"])
run(admin + ["apt-get", "install", "-y", "--no-install-recommends", "postgresql-17", "postgresql-client-17"])

arch = {"x86_64": "x64", "aarch64": "arm64"}[platform.machine()]
filename = f"node-v24.15.0-linux-{arch}.tar.xz"
base = "https://nodejs.org/dist/v24.15.0/"
with urlopen(base + "SHASUMS256.txt", timeout=60) as response:
    checksums = response.read().decode()
expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == filename)
archive = TOOLCHAIN / filename
with urlopen(base + filename, timeout=60) as response:
    archive.write_bytes(response.read())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node checksum mismatch"
with tarfile.open(archive) as bundle:
    bundle.extractall(TOOLCHAIN, filter="data")
node_bin = TOOLCHAIN / filename.removesuffix(".tar.xz") / "bin"
os.environ["PATH"] = str(node_bin) + os.pathsep + os.environ["PATH"]
run(["npm", "install", "--global", "--prefix", str(node_bin.parent), "pnpm@11.25.0"])
run(["node", "--version"])
run(["pnpm", "--version"])
run(["/usr/lib/postgresql/17/bin/pg_ctl", "--version"])


In [ ]:
if not ROOT.exists():
    run(["git", "clone", "https://github.com/znp0/loan-core.git", str(ROOT)])
run(["git", "fetch", "origin", SOURCE_REF], cwd=ROOT)
run(["git", "switch", "--detach", "FETCH_HEAD"], cwd=ROOT)
assert not list(ROOT.glob("apps/*/.env")), "Use a clean checkout without managed credentials"
run(["pnpm", "install", "--frozen-lockfile"], cwd=ROOT)
run(["pnpm", "exec", "turbo", "run", "build", "--filter=identity-service"], cwd=ROOT)
run(["git", "rev-parse", "HEAD"], cwd=ROOT)


## Exercise startup and clean shutdown

Use a fresh output directory on each attempt. The JSON report records hardware, cgroup CPU/RAM limits, the UI quota note, Internet checks, exact tools, source revision, health response, and shutdown results. A failed preflight exits with an error and still writes diagnostic artifacts. Download the ZIP even if a cell fails; do not interpret a failed or local run as Kaggle feasibility.


In [ ]:
from datetime import datetime, timezone

OUTPUT = Path("/kaggle/working") / ("loan-core-preflight-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
run(["python3", str(ROOT / "benchmarks/kaggle/preflight.py"), "--output", str(OUTPUT), "--quota-note", QUOTA_NOTE])


In [ ]:
import json
from IPython.display import FileLink, display

report = json.loads((OUTPUT / "report.json").read_text())
print(json.dumps({key: report[key] for key in ["status", "kaggle_feasibility_verified", "source_revision", "node", "pnpm", "postgres_server_version", "health", "service_shutdown", "database_shutdown"]}, indent=2))
display(FileLink(str(OUTPUT) + ".zip"))
